# CoRe-TFM — Clean replication and provenance capture (October 2026)

**Purpose:** Do the reproducibility work still missing after the 160-cache manuscript audit. This notebook is **new-GPU replication**, **not** another reanalysis and **not** a claim that old checkpoint bytes have been authenticated.

It uses the previous complete 160-cache ZIP as a **frozen reference**. It can perform offline integrity/provenance checks on CPU, capture official OpenML metadata and dataset-file bytes when connected to the internet, and rerun a *small* set of matched model configurations on a T4 GPU with explicit checkpoint paths/hashes.

**Never upload model checkpoint weights or authentication secrets to a public repository.** TabPFN-3 weights have a separate non-commercial license. No Google Drive is used. You must personally accept its license and set a secret.

The prior evidence archive is `CoReTFM_FULL160_Combined_Verified_Evidence.zip`, **not** the v7 audit-output ZIP.

In [ ]:
from pathlib import Path
import json, os, sys, hashlib, zipfile, subprocess, platform, importlib.metadata as im, time, io
import numpy as np

IN_COLAB='google.colab' in sys.modules
WORK=Path('/content/CoReTFM_CLEAN_REPLICATION' if IN_COLAB else '/mnt/data/coretfm_replication_cpu_smoketest')
WORK.mkdir(parents=True,exist_ok=True)
REFERENCE_ZIP=Path('/content/CoReTFM_FULL160_Combined_Verified_Evidence.zip' if IN_COLAB else '/mnt/data/CoReTFM_FULL160_Combined_Verified_Evidence.zip')
ASK_UPLOAD_IF_MISSING=True
RUN_ONLINE_OPENML_PROVENANCE=False  # Set True once network is available; records exact dataset version/file checksum
RUN_GPU_REPLICATION=False           # On Colab GPU set True, after agreeing to model licenses
MODEL_SELECTION=('tabiclv2','tabpfn3')
# Three distinct datasets, one split, one primary training size, two models = 6 new model configurations.
REPL_DATASETS=('car','nursery','diamonds')
REPL_SPLIT=(23,0)
REPL_CONTEXT=256
TABPFN_LICENSE_ACCEPTED=False       # True only after accepting Prior Labs license
INSTALL_MODEL_PACKAGES=True
DOWNLOAD_RESULT_ZIP=True
# Reference file SHA-256 from the publicly posted TabICLv2 weight file. Not proof of historical loading.
EXPECTED_TABICL_REMOTE_SHA256='bdc7dbd5e4ff21f8f0456fcf90c6b7cdf72dbea960f2d05b19bec19f9b3d4ed0'
print('Workspace:',WORK,'GPU work enabled:',RUN_GPU_REPLICATION)

In [ ]:
if not REFERENCE_ZIP.is_file() and IN_COLAB and ASK_UPLOAD_IF_MISSING:
    from google.colab import files
    print('Upload exactly: CoReTFM_FULL160_Combined_Verified_Evidence.zip')
    uploaded=files.upload()
    if 'CoReTFM_FULL160_Combined_Verified_Evidence.zip' not in uploaded:
        raise FileNotFoundError('Required full160 ZIP not uploaded (audit-output ZIP is not the input).')
    REFERENCE_ZIP=Path('/content/CoReTFM_FULL160_Combined_Verified_Evidence.zip')
if not REFERENCE_ZIP.is_file():
    raise FileNotFoundError('Missing input ZIP: '+str(REFERENCE_ZIP))
sha=lambda b:hashlib.sha256(b).hexdigest()
with zipfile.ZipFile(REFERENCE_ZIP) as z:
    assert z.testzip() is None,'ZIP CRC failure'
    names=set(z.namelist())
    assert 'COMBINED_SHA256_MANIFEST.json' in names,'Not the full160 combined ZIP'
    manifest=json.loads(z.read('COMBINED_SHA256_MANIFEST.json'))
    for item in manifest:
        b=z.read(item['path'])
        assert len(b)==item['bytes'] and sha(b)==item['sha256'],item['path']
    ref_caches=sorted(n for n in names if n.startswith('real_caches/') and n.endswith('.npz'))
    ref_splits=sorted(n for n in names if n.startswith('splits/') and n.endswith('.json'))
    assert len(ref_caches)==160 and len(ref_splits)==40,(len(ref_caches),len(ref_splits))
    sidecars=[json.loads(z.read(n)) for n in names if n.startswith('real_caches/') and n.endswith('.json')]
    historic_unverified=sum(m.get('checkpoint_sha256') in ('NOT_VERIFIED','',None) for m in sidecars)
    metadata={'reference_zip_sha256':sha(REFERENCE_ZIP.read_bytes()),'manifest_count':len(manifest),
              'cache_count':len(ref_caches),'frozen_splits':len(ref_splits),
              'historical_checkpoint_hash_unverified_count':historic_unverified}
    (WORK/'OFFLINE_FROZEN_EVIDENCE.json').write_text(json.dumps(metadata,indent=2))
print('Verified frozen evidence:',metadata)

## 1. Frozen evidence recovery — splits only

We restore **only split specifications**, never old model predictions into the new results folder. This makes it impossible to accidentally report an old cache as a new independent inference run.

In [ ]:
for sub in ('splits','datasets','real_caches','reports','checkpoints'):
    (WORK/sub).mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(REFERENCE_ZIP) as z:
    for n in ref_splits:
        target=WORK/n
        b=z.read(n)
        if target.exists() and sha(target.read_bytes())!=sha(b):raise ValueError('Conflicting frozen split: '+str(target))
        target.write_bytes(b)
print('Frozen splits restored:',len(ref_splits),'; no previous .npz caches copied.')

## 2. OpenML provenance — optional network step

Fetch metadata by **numeric OpenML dataset ID**, not dataset name. This records the API version and MD5 reported by OpenML. Where an original dataset download is available, compute a local SHA-256/MD5 and compare to the publisher-provided MD5. It does not silently equate processed pandas `.pkl` bytes with original OpenML ARFF bytes. Network failures are recorded explicitly.

In [ ]:
OPENML_IDS={'anneal':[46906],'credit':[46918],'phishing':[46963],'mic':[46980],
            'customer':[46938],'car':[40975],'marketing':[46940],'nursery':[26],
            'diamonds':[42225],'wine':[40691,40498]}
raw=[]
if RUN_ONLINE_OPENML_PROVENANCE:
    import requests
    for name,ids in OPENML_IDS.items():
        for oid in ids:
            row={'dataset':name,'openml_id':oid,'retrieval_status':'FAILED'}
            try:
                url=f'https://www.openml.org/api/v1/json/data/{oid}'
                r=requests.get(url,timeout=45);r.raise_for_status()
                j=r.json(); desc=j['data_set_description']
                row.update({'retrieval_status':'OK_METADATA','version':desc.get('version'),
                            'file_id':desc.get('file_id'),'name':desc.get('name'),
                            'md5_checksum_reported':desc.get('md5_checksum'),
                            'licence':desc.get('licence'),'url':desc.get('url'),
                            'metadata_json_sha256':sha(r.content)})
                (WORK/'reports'/f'OPENML_METADATA_{oid}.json').write_bytes(r.content)
                # Raw byte downloading is explicit: unlike a pandas pickle, this is the URL OpenML cites.
                raw_url=desc.get('url')
                if raw_url:
                    with requests.get(raw_url,timeout=120,stream=True) as dr:
                        dr.raise_for_status()
                        h=hashlib.sha256();hmd5=hashlib.md5();size=0
                        for chunk in dr.iter_content(1024*1024):
                            if chunk:h.update(chunk);hmd5.update(chunk);size+=len(chunk)
                    row.update({'original_download_sha256':h.hexdigest(), 'downloaded_bytes':size,
                                'raw_md5_matches_openml':hmd5.hexdigest().lower()==str(desc.get('md5_checksum','')).lower()})
                    row['retrieval_status']='OK_RAW_HASHED' if row['raw_md5_matches_openml'] else 'RAW_MD5_MISMATCH_OR_UNAVAILABLE'
            except Exception as e:
                row.update({'error_type':type(e).__name__,'error':str(e)[:220]})
            raw.append(row)
else:
    raw=[{'dataset':name,'openml_id':oid,'retrieval_status':'NOT_RUN_USER_DISABLED'} for name,ids in OPENML_IDS.items() for oid in ids]
(WORK/'reports'/'OPENML_DATA_PROVENANCE.json').write_text(json.dumps(raw,indent=2,default=str))
print('OpenML provenance statuses:',{s:sum(r['retrieval_status']==s for r in raw) for s in sorted(set(r['retrieval_status'] for r in raw))})

## 3. Embedded, frozen CoRe-TFM inference engine

The original four-view construction is embedded for compatibility with the saved caches. Its numerical procedures are not replaced by a different model or a new label split.

In [ ]:
import types
ENGINE_SOURCE = '"""CoRe-TFM resumable real-inference and evidence engine, protocol v2 (2026-10-19).\nStandalone implementation incorporating the paired-OpenML specification and four\nview construction from bnssaanirudh/core-tfm. New experiments must be analyzed\nseparately from the historical main-branch benchmark.\n"""\nfrom __future__ import annotations\nimport os, sys, json, hashlib, tempfile, time, traceback, gc, importlib.metadata as im\nfrom pathlib import Path\nfrom datetime import datetime, timezone\nfrom dataclasses import dataclass\nfrom typing import Any\nimport numpy as np\nimport pandas as pd\nfrom scipy.stats import wilcoxon\n\nPROTOCOL = "coretfm-v2-independent-fixed-test-2026-10-19"\nDATASETS = {\n    "anneal":(46906,"classes","steel"), "credit":(46918,"good_or_bad_customer","checking_status"),\n    "phishing":(46963,"WebsiteType","SFH"), "mic":(46980,"LET_IS","ASP_S_n"),\n    "customer":(46938,"bad_client_target","education"), "car":(40975,"class","safety"),\n    "marketing":(46940,"Response","Marital_Status"), "nursery":(26,"class","health"),\n    "diamonds":(42225,"cut","color"), "wine":([40691,40498],"class","color"),\n}\nCHECKPOINTS={"tabiclv2":"tabicl-classifier-v2-20260212.ckpt","tabpfn3":"ModelVersion.V3"}\n\n\ndef utcnow(): return datetime.now(timezone.utc).isoformat()\ndef sha_bytes(b): return hashlib.sha256(b).hexdigest()\ndef sha_obj(v): return sha_bytes(json.dumps(v,sort_keys=True,default=str).encode())\ndef _atomic_bytes(path:Path, data:bytes):\n    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)\n    with tempfile.NamedTemporaryFile(dir=path.parent,prefix=\'.tmp_\',delete=False) as f:\n        f.write(data); tmp=Path(f.name)\n    os.replace(tmp,path)\ndef put_json(path,payload): _atomic_bytes(Path(path),json.dumps(payload,indent=2,sort_keys=True,default=str).encode())\ndef get_json(path): return json.loads(Path(path).read_text())\ndef out_root():\n    custom=os.environ.get(\'COREFM_OUT\')\n    if custom: return Path(custom).expanduser().resolve()\n    mounted=Path(\'/content/drive/MyDrive\')\n    if mounted.exists() and mounted.is_dir(): return mounted/\'CoRe_TFM_Resumable_v2\'\n    return Path.cwd()/\'CoRe_TFM_Resumable_v2_results\'\ndef init_dirs(root=None):\n    root=Path(root) if root is not None else out_root()\n    for k in [\'datasets\',\'splits\',\'real_caches\',\'smoke_caches\',\'reports\',\'runtime\',\'failures\']:(root/k).mkdir(parents=True,exist_ok=True)\n    return root\n\ndef cuda_status():\n    try:\n        import torch\n        ok=bool(torch.cuda.is_available())\n        return {\'cuda\':ok,\'torch\':torch.__version__,\'gpu\':torch.cuda.get_device_name(0) if ok else None}\n    except Exception as e: return {\'cuda\':False,\'reason\':repr(e)}\n\ndef version(package):\n    try: return im.version(package)\n    except im.PackageNotFoundError: return \'not_installed\'\n\n\ndef load_pair_dataset(name,root=None,download=True):\n    """Pinned two-outcome OpenML pair; dataset bytes cached on Drive as pickle.\n    Missing values and derived features follow the repository\'s openml.py policy.\n    No label from validation/test ever enters model input other than hypothetical\n    conditioning values enumerated over every possible category.\n    """\n    if name not in DATASETS:raise KeyError(name)\n    root=init_dirs(root);loc=root/\'datasets\'/f\'{name}.pkl\'\n    if loc.exists():\n        data=pd.read_pickle(loc)\n        return data[\'X\'],data[\'a\'],data[\'b\'],data[\'source\']\n    if not download: raise FileNotFoundError(loc)\n    from sklearn.datasets import fetch_openml\n    ids,ta,tb=DATASETS[name]\n    if name==\'wine\':\n        cols=[\'fixed_acidity\',\'volatile_acidity\',\'citric_acid\',\'residual_sugar\',\'chlorides\',\'free_sulfur_dioxide\',\'total_sulfur_dioxide\',\'density\',\'pH\',\'sulphates\',\'alcohol\']\n        parts=[]\n        for oid,c in ((40691,\'red\'),(40498,\'white\')):\n            ds=fetch_openml(data_id=oid,as_frame=True,parser=\'auto\')\n            x=ds.data.copy();x.columns=cols;x[\'__a__\']=pd.Series(ds.target).to_numpy();x[\'__b__\']=c;parts.append(x)\n        frame=pd.concat(parts,ignore_index=True)\n        source={\'openml_ids\':[40691,40498],\'target_a\':ta,\'target_b\':tb}\n    else:\n        ds=fetch_openml(data_id=ids,as_frame=True,parser=\'auto\')\n        x=ds.data.copy()\n        if tb not in x:raise ValueError(f\'{name}: expected target B {tb} absent; columns={list(x.columns)[:15]}\')\n        if name==\'diamonds\':\n            if ta not in x: raise ValueError(\'Expected diamonds class A column cut missing\')\n            x[\'__a__\']=x.pop(ta)\n        else:\n            x[\'__a__\']=np.asarray(ds.target)\n        x[\'__b__\']=x.pop(tb)\n        if name==\'mic\':\n            features=x.drop(columns=[\'__a__\',\'__b__\'])\n            x=x.loc[:, list(features.columns[features.isna().mean()<=.02])+[\'__a__\',\'__b__\']]\n        if name==\'marketing\':\n            for col in x.columns:\n                if \'date\' in str(col).lower() or \'enroll\' in str(col).lower():\n                    parsed=pd.to_datetime(x[col],errors=\'coerce\',dayfirst=True)\n                    if parsed.notna().mean()>.95:\n                        x[col]=(parsed-parsed.min()).dt.days.astype(float);break\n        frame=x\n        source={\'openml_id\':ids,\'target_a\':ta,\'target_b\':tb}\n    frame=frame.dropna(axis=0).reset_index(drop=True)\n    a=frame.pop(\'__a__\').astype(\'category\'); b=frame.pop(\'__b__\').astype(\'category\')\n    X=frame.copy()\n    # NOTE: no leakage - targets are removed from predictors.\n    if X.columns.duplicated().any():raise ValueError(\'duplicate feature names\')\n    data={\'X\':X,\'a\':a,\'b\':b,\'source\':source}\n    pd.to_pickle(data,loc)\n    return X,a,b,source\n\n\ndef coverage_order(indices, a, b, rng):\n    """Deterministically frontload both marginal class coverage for nested caps."""\n    seq=np.array(indices,dtype=int).copy();rng.shuffle(seq)\n    all_a=set(a.iloc[seq].astype(str));all_b=set(b.iloc[seq].astype(str))\n    seen_a=set();seen_b=set();picks=[];unused=list(seq)\n    while (seen_a!=all_a or seen_b!=all_b) and unused:\n        gains=[(int(str(a.iloc[i]) not in seen_a)+int(str(b.iloc[i]) not in seen_b)) for i in unused]\n        pos=int(np.argmax(gains)); i=unused.pop(pos)\n        picks.append(i);seen_a.add(str(a.iloc[i]));seen_b.add(str(b.iloc[i]))\n    return np.array(picks+unused,dtype=int),len(picks)\n\n\ndef make_split(name,seed=23,fold=0,root=None,test_n=128,val_n=205):\n    from sklearn.model_selection import train_test_split\n    root=init_dirs(root);X,a,b,source=load_pair_dataset(name,root)\n    path=root/\'splits\'/f\'{name}__seed{seed}__fold{fold}.json\'\n    if path.exists():\n        p=get_json(path)\n        if (p.get(\'protocol\')!=PROTOCOL or p[\'n_total\']!=len(X) or p.get(\'requested_test_n\')!=test_n or p.get(\'requested_val_n\')!=val_n):\n            raise ValueError(\'split incompatible with protocol, dataset size or test/validation reservation; refusing overwrite\')\n        return p\n    if len(X)<test_n+val_n+65:raise ValueError(f\'{name}: insufficient rows for fixed test/validation split\')\n    idx=np.arange(len(X));r=seed*1009+fold*100003\n    # Deterministic random outer split; no stratification promise for very rare joint classes.\n    pool,test=train_test_split(idx,test_size=test_n,random_state=r,shuffle=True)\n    train_pool,val=train_test_split(pool,test_size=val_n,random_state=r+17,shuffle=True)\n    ordered,coverage=coverage_order(train_pool,a,b,np.random.default_rng(r+29))\n    vordered=np.array(val,dtype=int);np.random.default_rng(r+31).shuffle(vordered)\n    split={\'protocol\':PROTOCOL,\'dataset\':name,\'seed\':seed,\'fold\':fold,\'n_total\':len(X),\'source\':source,\n           \'train_pool_ids\':ordered.tolist(),\'val_ids\':vordered.tolist(),\'test_ids\':np.array(test,dtype=int).tolist(),\n           \'coverage_frontload_rows\':coverage,\'requested_test_n\':test_n,\'requested_val_n\':val_n,\'classes_a\':list(map(str,a.cat.categories)),\n           \'classes_b\':list(map(str,b.cat.categories))}\n    assert_no_overlap(split)\n    split[\'split_sha256\']=sha_obj({k:v for k,v in split.items() if k!=\'split_sha256\'})\n    put_json(path,split);return split\n\ndef assert_no_overlap(s):\n    i=[set(map(int,s[k])) for k in [\'train_pool_ids\',\'val_ids\',\'test_ids\']]\n    if any(len(set(s[k]))!=len(s[k]) for k in [\'train_pool_ids\',\'val_ids\',\'test_ids\']):raise AssertionError(\'duplicate IDs within split\')\n    if i[0]&i[1] or i[0]&i[2] or i[1]&i[2]:raise AssertionError(\'train/val/test overlap\')\n\ndef _condition_col(X,name,values,value):\n    v=X.copy(); v[name]=pd.Categorical([str(value)]*len(v),categories=list(map(str,values)))\n    return v\n\ndef _aligned_proba(est,X,classes):\n    pred=np.asarray(est.predict_proba(X),dtype=float)\n    order={str(c):i for i,c in enumerate(np.asarray(est.classes_).tolist())}\n    if set(order)!=set(map(str,classes)):raise RuntimeError(\'model classes differ from the frozen global label domain\')\n    pred=pred[:,[order[str(c)] for c in classes]]\n    if pred.shape!=(len(X),len(classes)) or not np.isfinite(pred).all():raise ValueError(\'invalid model predict_proba\')\n    if np.min(pred)<-1e-9 or np.max(pred)>1+1e-9:raise ValueError(\'out-of-range probability\')\n    pred=np.maximum(pred,0)\n    if (pred.sum(axis=1)<=0).any():raise ValueError(\'zero probability mass\')\n    return pred/pred.sum(axis=1,keepdims=True)\n\ndef make_factory(model,device=\'cuda\',seed=23):\n    if model==\'tabiclv2\':\n        from tabicl import TabICLClassifier\n        def f():return TabICLClassifier(n_estimators=2,kv_cache=False,batch_size=2,checkpoint_version=CHECKPOINTS[\'tabiclv2\'],device=device,random_state=seed)\n        return f\n    if model==\'tabpfn3\':\n        from tabpfn import TabPFNClassifier\n        from tabpfn.constants import ModelVersion\n        if not hasattr(ModelVersion,\'V3\'):raise RuntimeError(\'TabPFN V3 not supported by installed package\')\n        def f():\n            obj=TabPFNClassifier.create_default_for_version(ModelVersion.V3)\n            p=obj.get_params();changes={k:v for k,v in {\'n_estimators\':2,\'device\':device,\'random_state\':seed}.items() if k in p}\n            if changes:obj.set_params(**changes)\n            return obj\n        return f\n    if model==\'smoke_logreg\':\n        from sklearn.compose import ColumnTransformer\n        from sklearn.preprocessing import OneHotEncoder,StandardScaler\n        from sklearn.impute import SimpleImputer\n        from sklearn.pipeline import make_pipeline\n        from sklearn.linear_model import LogisticRegression\n        def f():\n            # Fully self-contained smoke model; never publish as a TFM run.\n            return make_pipeline(ColumnTransformer([(\'category\',make_pipeline(SimpleImputer(strategy=\'most_frequent\'),OneHotEncoder(handle_unknown=\'ignore\')),lambda df:list(df.select_dtypes(exclude=\'number\').columns)),(\'numeric\',make_pipeline(SimpleImputer(strategy=\'median\'),StandardScaler()),lambda df:list(df.select_dtypes(include=\'number\').columns))],remainder=\'drop\'),LogisticRegression(max_iter=300))\n        return f\n    raise ValueError(f\'Unknown model: {model}\')\n\ndef four_views(factory,Xtrain,atrain,btrain,Xval,Xtest,classes_a,classes_b):\n    """Fit four estimators with distinct views; return all arrays and fit/predict timings.\n    Only training labels enter conditional training features. For prediction each\n    possible conditioning value is enumerated; test ground truth is never used.\n    """\n    nva=len(Xval);nte=len(Xtest);ka=len(classes_a);kb=len(classes_b)\n    trainA=pd.Series(pd.Categorical(atrain.astype(str),categories=classes_a))\n    trainB=pd.Series(pd.Categorical(btrain.astype(str),categories=classes_b))\n    Xa=Xtrain.reset_index(drop=True).copy();V=Xval.reset_index(drop=True);T=Xtest.reset_index(drop=True)\n    if any(c in Xa for c in [\'__cond_A__\',\'__cond_B__\']):raise ValueError(\'reserved feature collision\')\n    views={};times={};\n    def view(key,y,extra_col=None,extra_vals=None,condition_on=None):\n        if extra_col is None:\n            train=Xa; xvs=[V];xts=[T]\n        else:\n            train=_condition_col(Xa,extra_col,extra_vals,np.nan) # overwritten below\n            train[extra_col]=pd.Categorical(np.asarray(condition_on).astype(str),categories=list(map(str,extra_vals)))\n            xvs=[_condition_col(V,extra_col,extra_vals,c) for c in extra_vals]\n            xts=[_condition_col(T,extra_col,extra_vals,c) for c in extra_vals]\n        start=time.perf_counter();est=factory();est.fit(train,y);fit_secs=time.perf_counter()-start\n        pred_start=time.perf_counter()\n        expected=classes_a if key in (\'a\',\'a_given_b\') else classes_b\n        pv=np.stack([_aligned_proba(est,x,expected) for x in xvs],axis=1)\n        pt=np.stack([_aligned_proba(est,x,expected) for x in xts],axis=1)\n        inference_secs=time.perf_counter()-pred_start\n        if pv.shape[1]==1:pv=pv[:,0,:];pt=pt[:,0,:]\n        views[key+\'_val\']=pv;views[key+\'_test\']=pt\n        times[key]={\'fit_with_load_sec\':fit_secs,\'predict_sec\':inference_secs,\'n_predict_calls\':len(xvs)+len(xts)}\n        del est;gc.collect()\n        try:\n            import torch\n            if torch.cuda.is_available():torch.cuda.empty_cache()\n        except ImportError:pass\n    view(\'a\',trainA)\n    view(\'b\',trainB)\n    view(\'a_given_b\',trainA,extra_col=\'__cond_B__\',extra_vals=classes_b,condition_on=trainB)\n    view(\'b_given_a\',trainB,extra_col=\'__cond_A__\',extra_vals=classes_a,condition_on=trainA)\n    assert views[\'a_given_b_val\'].shape==(nva,kb,ka)\n    assert views[\'b_given_a_val\'].shape==(nva,ka,kb)\n    assert views[\'a_given_b_test\'].shape==(nte,kb,ka)\n    return views,times\n\ndef _validate_probability_view(arr,axis=-1):\n    arr=np.asarray(arr,dtype=float)\n    if not np.all(np.isfinite(arr)) or (arr<-1e-8).any():raise ValueError(\'invalid probabilities\')\n    if not np.allclose(arr.sum(axis=axis),1,atol=1e-4):raise ValueError(\'probability rows not normalized\')\n\ndef run_case(dataset,model,seed=23,fold=0,train_cap=64,root=None,test_n=128,val_n=205,allow_cpu=False):\n    root=init_dirs(root)\n    if model!=\'smoke_logreg\' and not cuda_status()[\'cuda\'] and not allow_cpu:\n        return {\'status\':\'SKIPPED_NO_GPU\',\'dataset\':dataset,\'model\':model,\'explanation\':\'Run in a CUDA Colab session, or explicitly allow CPU (slow).\'}\n    split=make_split(dataset,seed,fold,root,test_n=test_n,val_n=val_n)\n    assert_no_overlap(split)\n    ids=np.asarray(split[\'train_pool_ids\'],dtype=int)\n    if train_cap>len(ids):return {\'status\':\'SKIPPED_INSUFFICIENT_TRAIN_POOL\',\'train_cap\':train_cap,\'train_pool_size\':len(ids)}\n    if train_cap<split[\'coverage_frontload_rows\']:\n        return {\'status\':\'SKIPPED_CLASS_COVERAGE\',\'train_cap\':train_cap,\'minimum_required\':split[\'coverage_frontload_rows\']}\n    filekey=f\'{dataset}__{model}__seed{seed}__fold{fold}__train{train_cap}\'\n    cdir=root/(\'smoke_caches\' if model==\'smoke_logreg\' else \'real_caches\')\n    path=cdir/(filekey+\'.npz\')\n    if path.exists():\n        try:\n            _,meta=read_cache(path)\n            if meta[\'split_sha256\']==split[\'split_sha256\'] and meta[\'protocol\']==PROTOCOL and meta[\'train_cap\']==train_cap and meta[\'model\']==model:\n                pkg={\'tabiclv2\':\'tabicl\',\'tabpfn3\':\'tabpfn\',\'smoke_logreg\':\'scikit-learn\'}[model]\n                current=version(pkg)\n                if current not in (\'not_installed\',meta[\'package_version\']):\n                    return {\'status\':\'CACHED_VERSION_CONFLICT\',\'path\':str(path),\'cached_package_version\':meta[\'package_version\'],\'current_package_version\':current}\n                return {\'status\':\'CACHED_VERIFIED\',\'path\':str(path)}\n        except Exception as ex:raise RuntimeError(f\'Existing cache is corrupt or incompatible; DO NOT overwrite: {path}: {ex}\')\n    X,a,b,source=load_pair_dataset(dataset,root)\n    tr=ids[:train_cap];va=np.array(split[\'val_ids\'],dtype=int);te=np.array(split[\'test_ids\'],dtype=int)\n    ca=split[\'classes_a\'];cb=split[\'classes_b\']\n    if set(map(str,a.iloc[tr]))!=set(ca) or set(map(str,b.iloc[tr]))!=set(cb):\n        return {\'status\':\'SKIPPED_CLASS_COVERAGE\',\'dataset\':dataset,\'train_cap\':train_cap}\n    first=time.perf_counter()\n    arrays,times=four_views(make_factory(model,device=\'cuda\' if cuda_status()[\'cuda\'] else \'cpu\',seed=seed),X.iloc[tr],a.iloc[tr].reset_index(drop=True),b.iloc[tr].reset_index(drop=True),X.iloc[va],X.iloc[te],ca,cb)\n    for k,v in arrays.items():_validate_probability_view(v)\n    lookup_a={c:i for i,c in enumerate(ca)};lookup_b={c:i for i,c in enumerate(cb)}\n    for role,ix in [(\'val\',va),(\'test\',te)]:\n        arrays[f\'ya_{role}\']=np.array([lookup_a[str(v)] for v in a.iloc[ix]],dtype=np.int32)\n        arrays[f\'yb_{role}\']=np.array([lookup_b[str(v)] for v in b.iloc[ix]],dtype=np.int32)\n    for role,ix in [(\'train\',tr),(\'val\',va),(\'test\',te)]:arrays[role+\'_ids\']=np.array(ix,dtype=np.int32)\n    meta={\'protocol\':PROTOCOL,\'dataset\':dataset,\'model\':model,\'artifact_kind\':\'smoke_only\' if model==\'smoke_logreg\' else \'real_tfm\',\n          \'seed\':seed,\'fold\':fold,\'train_cap\':train_cap,\'n_train\':len(tr),\'n_val\':len(va),\'n_test\':len(te),\n          \'split_sha256\':split[\'split_sha256\'],\'train_ids_sha\':sha_obj(tr.tolist()),\'val_ids_sha\':sha_obj(va.tolist()),\n          \'test_ids_sha\':sha_obj(te.tolist()),\'classes_a\':ca,\'classes_b\':cb,\n          \'package_version\':version({\'tabiclv2\':\'tabicl\',\'tabpfn3\':\'tabpfn\',\'smoke_logreg\':\'scikit-learn\'}[model]),\n          \'checkpoint_identifier\':CHECKPOINTS.get(model,\'not_a_TFM\'),\n          \'checkpoint_sha256\':\'NOT_VERIFIED\',\'checkpoint_revision_confirmed\':False,\n          \'device\':cuda_status(),\'timing\':times,\'elapsed_s\':time.perf_counter()-first,\'created_utc\':utcnow(),\n          \'source\':source,\'n_probabilities\':sum(np.size(x) for x in arrays.values()),\n          \'evidence_caveat\':\'Checkpoint filename recorded; actual weight-byte SHA256 not obtained.\'}\n    with tempfile.NamedTemporaryFile(dir=cdir,prefix=\'.npz_tmp_\',suffix=\'.npz\',delete=False) as f:tmp=Path(f.name)\n    try:\n        np.savez_compressed(tmp,**arrays)\n        with np.load(tmp,allow_pickle=False) as test:\n            if set(test.files)!=set(arrays):raise AssertionError(\'NPZ key mismatch\')\n        meta[\'npz_sha256\']=sha_bytes(tmp.read_bytes())\n        os.replace(tmp,path)\n        put_json(path.with_suffix(\'.json\'),meta)\n    finally:\n        if tmp.exists():tmp.unlink()\n    return {\'status\':\'REAL_MODEL_CACHE_WRITTEN\' if model!=\'smoke_logreg\' else \'SMOKE_ONLY_CACHE_WRITTEN\',\'path\':str(path),\'n_train\':len(tr),\'n_val\':len(va),\'n_test\':len(te),\'elapsed_sec\':meta[\'elapsed_s\']}\n\ndef read_cache(path):\n    path=Path(path);side=path.with_suffix(\'.json\')\n    if not side.exists():raise FileNotFoundError(f\'Cache sidecar missing: {side}\')\n    meta=get_json(side)\n    if meta.get(\'npz_sha256\')!=sha_bytes(path.read_bytes()):raise ValueError(f\'Cache checksum mismatch: {path}\')\n    if meta.get(\'protocol\')!=PROTOCOL:raise ValueError(\'incompatible protocol\')\n    with np.load(path,allow_pickle=False) as npz: data={k:npz[k] for k in npz.files}\n    expected=[\'a_val\',\'b_val\',\'a_given_b_val\',\'b_given_a_val\',\'a_test\',\'b_test\',\'a_given_b_test\',\'b_given_a_test\',\'ya_val\',\'yb_val\',\'ya_test\',\'yb_test\',\'train_ids\',\'val_ids\',\'test_ids\']\n    if set(expected)-set(data):raise ValueError(\'missing probability arrays/labels/ids\')\n    if len(data[\'a_val\'])!=meta[\'n_val\'] or len(data[\'a_test\'])!=meta[\'n_test\']:raise ValueError(\'cache length mismatch\')\n    for k in [\'a_val\',\'b_val\',\'a_given_b_val\',\'b_given_a_val\',\'a_test\',\'b_test\',\'a_given_b_test\',\'b_given_a_test\']:_validate_probability_view(data[k])\n    if len(set(data[\'train_ids\']) & set(data[\'val_ids\'])) or len(set(data[\'train_ids\']) & set(data[\'test_ids\'])) or len(set(data[\'test_ids\']) & set(data[\'val_ids\'])):raise ValueError(\'cache train/test leakage\')\n    if sha_obj(data[\'test_ids\'].tolist())!=meta[\'test_ids_sha\']:raise ValueError(\'test ids hash invalid\')\n    if sha_obj(data[\'val_ids\'].tolist())!=meta[\'val_ids_sha\']:raise ValueError(\'validation ids hash invalid\')\n    return data,meta\n\ndef normalize(x):\n    x=np.maximum(np.asarray(x,dtype=float),1e-12)\n    return x/x.sum(axis=(1,2),keepdims=True)\n\ndef implied(data,suffix):\n    pa=data[\'a_\'+suffix];pb=data[\'b_\'+suffix];pab=data[\'a_given_b_\'+suffix];pba=data[\'b_given_a_\'+suffix]\n    j1=np.transpose(pab,(0,2,1))*pb[:,None,:]\n    j2=pa[:,:,None]*pba\n    return normalize(j1),normalize(j2)\n\ndef hard_project(j,pa,pb,max_iter=150,tol=1e-9):\n    q=normalize(j);pa=np.maximum(pa,1e-12);pa=pa/pa.sum(axis=1,keepdims=True);pb=np.maximum(pb,1e-12);pb=pb/pb.sum(axis=1,keepdims=True)\n    for _ in range(max_iter):\n        q=q*(pa[:,:,None]/np.maximum(q.sum(axis=2,keepdims=True),1e-15))\n        q=q*(pb[:,None,:]/np.maximum(q.sum(axis=1,keepdims=True),1e-15))\n        if max(np.max(np.abs(q.sum(axis=2)-pa)),np.max(np.abs(q.sum(axis=1)-pb)))<tol:break\n    return normalize(q)\n\ndef candidates(data,suffix):\n    j1,j2=implied(data,suffix)\n    arith=normalize(.5*j1+.5*j2)\n    geo=normalize(np.sqrt(np.maximum(j1,1e-12)*np.maximum(j2,1e-12)))\n    hard=hard_project(arith,data[\'a_\'+suffix],data[\'b_\'+suffix])\n    return {\'arithmetic\':arith,\'raw_BA\':j1,\'raw_AB\':j2,\'geometric\':geo,\'hard\':hard,\n            \'soft_25\':normalize(.75*arith+.25*hard),\'soft_50\':normalize(.5*arith+.5*hard),\n            \'soft_75\':normalize(.25*arith+.75*hard)}\n\ndef per_sample_metrics(j,ya,yb):\n    flat=j.reshape(len(j),-1);ka,kb=j.shape[1:]\n    y=np.asarray(ya)*kb+np.asarray(yb)\n    nll=-np.log(np.maximum(flat[np.arange(len(y)),y],1e-12))\n    brier=(flat*flat).sum(axis=1)+1-2*flat[np.arange(len(y)),y]\n    return nll,brier\n\ndef select_method(cand_val,ya,yb,n_val,policy=\'full\',beta=.5):\n    m=min(int(n_val),len(ya))\n    if m<4:raise ValueError(\'too few validation labels\')\n    keys={\'pool\':[\'arithmetic\',\'geometric\'],\'full\':list(cand_val),\'regularized\':list(cand_val)}[policy]\n    losses={k:float(per_sample_metrics(cand_val[k][:m],ya[:m],yb[:m])[0].mean()) for k in keys}\n    best=min(losses,key=losses.get)\n    if policy==\'regularized\':\n        # Predeclared finite-validation guard; no test tuning and no confidence claim.\n        pen=beta*np.sqrt(2*np.log(max(len(keys),2))/m)\n        if losses[\'arithmetic\']-losses[best]<=pen:best=\'arithmetic\'\n    return best,losses\n\ndef score_cache(path,budgets=(16,32,52,103,205)):\n    d,meta=read_cache(path)\n    if meta.get(\'artifact_kind\')!=\'real_tfm\':raise ValueError(\'Smoke cache is not scientific real-TFM evidence\')\n    V=candidates(d,\'val\');T=candidates(d,\'test\')\n    yaV=d[\'ya_val\'];ybV=d[\'yb_val\'];yaT=d[\'ya_test\'];ybT=d[\'yb_test\']\n    j1,j2=implied(d,\'test\')\n    tv=float((.5*np.abs(j1-j2).sum(axis=(1,2))).mean())\n    rows=[]\n    for v in budgets:\n        if v>len(yaV):continue\n        for policy in [\'pool\',\'full\',\'regularized\']:\n            chosen,vloss=select_method(V,yaV,ybV,v,policy=policy)\n            nll,brier=per_sample_metrics(T[chosen],yaT,ybT)\n            pool_nll,pool_brier=per_sample_metrics(T[\'arithmetic\'],yaT,ybT)\n            rows.append({\'dataset\':meta[\'dataset\'],\'model\':meta[\'model\'],\'seed\':meta[\'seed\'],\'fold\':meta[\'fold\'],\n                         \'train_cap\':meta[\'train_cap\'],\'val_budget\':int(v),\'policy\':policy,\'selected\':chosen,\n                         \'joint_nll\':float(nll.mean()),\'joint_brier\':float(brier.mean()),\n                         \'pool_nll\':float(pool_nll.mean()),\'pool_brier\':float(pool_brier.mean()),\n                         \'delta_nll\':float((nll-pool_nll).mean()),\'delta_brier\':float((brier-pool_brier).mean()),\n                         \'factor_tv\':tv,\'n_val\':len(yaV),\'n_test\':len(yaT),\'test_ids_sha\':meta[\'test_ids_sha\'],\n                         \'split_sha256\':meta[\'split_sha256\'],\'cache_file\':str(path),\n                         \'checkpoint_revision_confirmed\':meta[\'checkpoint_revision_confirmed\'],\n                         \'checkpoint_sha256\':meta[\'checkpoint_sha256\']})\n    # Additional direct marginal and conditional diagnostics (proper scores).\n    marginal=[]\n    for label,direct,cond in [(\'A\',d[\'a_test\'],d[\'a_given_b_test\'][np.arange(len(yaT)),ybT,:]),\n                              (\'B\',d[\'b_test\'],d[\'b_given_a_test\'][np.arange(len(yaT)),yaT,:])]:\n        y=yaT if label==\'A\' else ybT\n        for kind,p in [(\'direct\',direct),(\'conditional\',cond)]:\n            nll=-np.log(np.maximum(p[np.arange(len(y)),y],1e-12))\n            brier=(p*p).sum(axis=1)+1-2*p[np.arange(len(y)),y]\n            marginal.append({\'dataset\':meta[\'dataset\'],\'model\':meta[\'model\'],\'seed\':meta[\'seed\'],\'fold\':meta[\'fold\'],\n                             \'train_cap\':meta[\'train_cap\'],\'target\':label,\'view\':kind,\'nll\':float(nll.mean()),\'brier\':float(brier.mean())})\n    return rows,marginal\n\ndef analyze_real(root=None,budgets=(16,32,52,103,205)):\n    root=init_dirs(root); paths=sorted((root/\'real_caches\').glob(\'*.npz\'))\n    if not paths:\n        result={\'status\':\'BLOCKED_NO_REAL_CACHE\',\'real_cache_count\':0,\'explanation\':\'Run NB36 or NB37 in GPU Colab first.\'}\n        put_json(root/\'reports\'/\'analysis_status.json\',result);return result\n    rows=[];marginal=[];bad=[]\n    for p in paths:\n        try:\n            r,m=score_cache(p,budgets);rows.extend(r);marginal.extend(m)\n        except Exception as e:bad.append({\'path\':str(p),\'error\':str(e)})\n    if not rows:\n        status={\'status\':\'BLOCKED_NO_VALID_CACHE\',\'errors\':bad}\n        put_json(root/\'reports\'/\'analysis_status.json\',status);return status\n    df=pd.DataFrame(rows);m=pd.DataFrame(marginal)\n    # For a given outer split/seed/fold the test IDs MUST stay fixed across caps and models.\n    same=df.groupby([\'dataset\',\'seed\',\'fold\'])[\'test_ids_sha\'].nunique()\n    if (same>1).any():raise AssertionError(f\'fixed test violation: {same[same>1].to_dict()}\')\n    df.to_csv(root/\'reports\'/\'fixed_test_factorial.csv\',index=False)\n    m.to_csv(root/\'reports\'/\'marginal_conditional_scores.csv\',index=False)\n    # Dataset-blocked inference: collapse folds and seeds/models within dataset.\n    blocks=df.groupby([\'dataset\',\'train_cap\',\'val_budget\',\'policy\'],as_index=False).agg(delta_nll=(\'delta_nll\',\'mean\'),delta_brier=(\'delta_brier\',\'mean\'))\n    blocks.to_csv(root/\'reports\'/\'dataset_blocks.csv\',index=False)\n    inf=[]\n    rng=np.random.default_rng(20261019)\n    for key,g in blocks.groupby([\'train_cap\',\'val_budget\',\'policy\']):\n        deltas=g.delta_nll.to_numpy();n=len(deltas)\n        ci=(np.nan,np.nan);p=np.nan\n        if n>=2:\n            bootstrap=rng.choice(deltas,size=(3000,n),replace=True).mean(axis=1)\n            ci=np.quantile(bootstrap,[.025,.975]);\n        if n>=5 and np.any(np.abs(deltas)>1e-12):\n            try:p=float(wilcoxon(deltas,alternative=\'two-sided\').pvalue)\n            except ValueError:pass\n        inf.append(dict(zip([\'train_cap\',\'val_budget\',\'policy\'],key))|{\'datasets\':n,\'mean_delta_nll\':float(deltas.mean()),\'ci95_low\':float(ci[0]),\'ci95_high\':float(ci[1]),\'wilcoxon_p_exploratory\':p,\n                          \'inferential_note\':\'Exploratory unless independent datasets are sufficient; hyperparameters fixed before test.\'})\n    pd.DataFrame(inf).to_csv(root/\'reports\'/\'dataset_blocked_uncertainty.csv\',index=False)\n    result={\'status\':\'ANALYZED_REAL_CACHES\',\'real_cache_count\':len(paths),\'valid_real_cache_count\':len(paths)-len(bad),\'invalid_cache_count\':len(bad),\n            \'method_rows\':len(df),\'datasets\':int(df.dataset.nunique()),\'models\':sorted(df.model.unique().tolist()),\'errors\':bad,\n            \'reports_dir\':str(root/\'reports\'),\'caveat\':\'Exploratory pilot and model weights not authenticated unless checkpoint hashes are separately verified.\'}\n    put_json(root/\'reports\'/\'analysis_status.json\',result);return result\n\ndef summarize_runtime(root=None):\n    root=init_dirs(root);rows=[]\n    for path in sorted((root/\'real_caches\').glob(\'*.npz\')):\n        try:\n            _,meta=read_cache(path)\n            t=meta[\'timing\']; direct=sum(t[k][\'fit_with_load_sec\']+t[k][\'predict_sec\'] for k in [\'a\',\'b\'])\n            four=direct+sum(t[k][\'fit_with_load_sec\']+t[k][\'predict_sec\'] for k in [\'a_given_b\',\'b_given_a\'])\n            rows.append({\'dataset\':meta[\'dataset\'],\'model\':meta[\'model\'],\'seed\':meta[\'seed\'],\'fold\':meta[\'fold\'],\n                         \'train_cap\':meta[\'train_cap\'],\'direct_two_view_wall_s\':direct,\n                         \'four_view_wall_s\':four,\'four_to_two_ratio\':four/direct if direct else np.nan,\n                         \'n_test\':meta[\'n_test\'],\'timing_type\':\'sequential end-to-end estimator fit+predict; includes possible checkpoint loading/download\',\n                         \'gpu\':meta[\'device\'].get(\'gpu\')})\n        except Exception as e: rows.append({\'cache\':str(path),\'error\':str(e)})\n    if rows:pd.DataFrame(rows).to_csv(root/\'runtime\'/\'cache_runtime.csv\',index=False)\n    return {\'status\':\'RUNTIME_SUMMARY\' if rows else \'NOT_RUN_NO_REAL_CACHES\',\'rows\':len(rows),\'path\':str(root/\'runtime\'/\'cache_runtime.csv\')}\n\ndef controlled_stress(root=None,repeats=8,start_repeat=0,validation_sizes=(8,16,32,64,128,256)):\n    """Checkpointed CPU known-truth study, a small number of repeats per call.\n    A new call appends only unique (regime,repeat,n_validation) cases. Deterministic\n    per-repeat seeds make results independent of execution interruption/batching.\n    """\n    root=init_dirs(root)\n    n=500; true=np.repeat(np.array([[[.65,.10],[.10,.15]]]),n,axis=0)\n    rows=[]\n    regimes=[\'marginal_reliable\',\'marginal_noisy\',\'shifted_validation\']\n    for regime_idx,regime in enumerate(regimes):\n        for r in range(start_repeat,start_repeat+repeats):\n            rng=np.random.default_rng(20261019+10001*r+regime_idx)\n            noise=rng.normal(0,.13,(n,2,2));v1=normalize(np.maximum(true+noise,1e-5));v2=normalize(np.maximum(true-noise,1e-5))\n            pooled=normalize(.5*v1+.5*v2)\n            if regime==\'marginal_reliable\':\n                pa=true.sum(axis=2);pb=true.sum(axis=1)\n            else:\n                fake=normalize(np.maximum(true+rng.normal(0,.16,(n,2,2)),1e-5))\n                pa=fake.sum(axis=2);pb=fake.sum(axis=1)\n            hard=hard_project(pooled,pa,pb)\n            for vb in validation_sizes:\n                rng_idx=np.random.default_rng(300001+1001*r+vb+regime_idx*100000)\n                yval=rng_idx.choice(4,size=vb,p=true[0].reshape(-1))\n                ytest=rng_idx.choice(4,size=200,p=true[0].reshape(-1))\n                va_pool=pooled[:vb];va_hard=hard[:vb]\n                if regime==\'shifted_validation\':va_hard=normalize(.8*va_hard+.2*np.roll(va_hard,1,axis=1))\n                score_pool=-np.log(va_pool.reshape(vb,4)[np.arange(vb),yval]).mean()\n                score_hard=-np.log(va_hard.reshape(vb,4)[np.arange(vb),yval]).mean()\n                chosen=\'hard\' if score_hard<score_pool else \'pool\'\n                ptest=(hard if chosen==\'hard\' else pooled)[-200:]\n                loss=-np.log(ptest.reshape(200,4)[np.arange(200),ytest]).mean()\n                loss_pool=-np.log(pooled[-200:].reshape(200,4)[np.arange(200),ytest]).mean()\n                rows.append({\'regime\':regime,\'repeat\':r,\'n_validation\':vb,\'chosen\':chosen,\n                             \'selected_minus_pool_nll\':float(loss-loss_pool),\'selected_nll\':float(loss),\'pool_nll\':float(loss_pool)})\n    new_df=pd.DataFrame(rows);dest=root/\'reports\'/\'SYNTHETIC_only_stress.csv\'\n    if dest.exists():\n        prior=pd.read_csv(dest)\n        new_df=pd.concat([prior,new_df],ignore_index=True).drop_duplicates([\'regime\',\'repeat\',\'n_validation\'],keep=\'last\')\n    new_df.sort_values([\'regime\',\'repeat\',\'n_validation\']).to_csv(dest,index=False)\n    return {\'status\':\'SYNTHETIC_COMPLETE\',\'new_rows\':len(rows),\'cumulative_rows\':len(new_df),\'path\':str(dest),\'not_real_model_evidence\':True}\n\ndef final_archive(root=None,strict=True):\n    root=init_dirs(root);cache_paths=sorted((root/\'real_caches\').glob(\'*.npz\'))\n    valid=[];invalid=[];datasets=set();models=set()\n    for p in cache_paths:\n        try:\n            _,meta=read_cache(p)\n            if meta[\'artifact_kind\']!=\'real_tfm\':raise ValueError(\'not a real TFM cache\')\n            valid.append(str(p));datasets.add(meta[\'dataset\']);models.add(meta[\'model\'])\n        except Exception as e:invalid.append({\'file\':str(p),\'reason\':str(e)})\n    reports=[\'analysis_status.json\',\'fixed_test_factorial.csv\',\'marginal_conditional_scores.csv\',\'dataset_blocks.csv\',\'dataset_blocked_uncertainty.csv\']\n    report_ok=all((root/\'reports\'/s).exists() for s in reports)\n    # Archive readiness deliberately stricter than successful notebook execution.\n    complete=bool(len(valid)>=8 and len(datasets)>=3 and {\'tabiclv2\',\'tabpfn3\'}<=models and report_ok and not invalid)\n    status={\'protocol\':PROTOCOL,\'status\':\'COMPLETE_PILOT_GATE\' if complete else \'PARTIAL_EVIDENCE_ONLY\',\n            \'valid_cache_count\':len(valid),\'datasets\':sorted(datasets),\'models\':sorted(models),\n            \'invalid\':invalid,\'report_files_complete\':report_ok,\n            \'complete_pilot_gate\':complete,\'full_paper_submission_ready\':False,\n            \'checkpoint_bytes_authenticated\':False,\n            \'notes\':\'Passing this modest pilot gate is not a full manuscript replication. Verify checkpoint weights and 10-dataset five-fold statistics separately.\'}\n    import zipfile\n    zip_path=root/\'reports\'/(\'CoReTFM_V2_PILOT_COMPLETE.zip\' if complete else \'CoReTFM_V2_PARTIAL_EVIDENCE.zip\')\n    status[\'zip_path\']=str(zip_path)\n    put_json(root/\'reports\'/\'EVIDENCE_GATE.json\',status)\n    # Archive outputs but exclude dataset pickle caches, secrets, model weights and temp files.\n    with zipfile.ZipFile(zip_path,\'w\',compression=zipfile.ZIP_DEFLATED,compresslevel=5) as z:\n        items=[]\n        for folder in [\'splits\',\'real_caches\',\'smoke_caches\',\'reports\',\'runtime\',\'failures\']:\n            for p in sorted((root/folder).rglob(\'*\')):\n                if p.is_file() and p.resolve()!=zip_path.resolve() and p.suffix.lower() not in (\'.ckpt\',\'.safetensors\',\'.token\',\'.zip\'):\n                    items.append(p)\n        for p in items:z.write(p,p.relative_to(root).as_posix())\n        manifest=[{\'path\':str(p.relative_to(root)),\'sha256\':sha_bytes(p.read_bytes()),\'bytes\':p.stat().st_size} for p in items]\n        z.writestr(\'ARCHIVE_SHA256_MANIFEST.json\',json.dumps(manifest,indent=2))\n    return status\n'
q=types.ModuleType('coretfm_replication_engine')
exec(compile(ENGINE_SOURCE,'coretfm_frozen_engine.py','exec'),q.__dict__)
assert q.PROTOCOL=='coretfm-v2-independent-fixed-test-2026-10-19'
print('Engine loaded:',q.PROTOCOL)

## 4. Inspect frozen labels and source data **before** GPU inference

Each rerun must regenerate the dataset from its recorded OpenML ID and obtain the same class domains and ordered validation/test labels as the original cache. Failure stops a job, rather than attempting to salvage mismatched training data. This step may download OpenML datasets once.

In [ ]:
import pandas as pd
REF_METADATA={}
for dataset in REPL_DATASETS:
    for model in MODEL_SELECTION:
        key=f'real_caches/{dataset}__{model}__seed{REPL_SPLIT[0]}__fold{REPL_SPLIT[1]}__train{REPL_CONTEXT}'
        REF_METADATA[(dataset,model)]=key

def align_to_frozen(dataset,model):
    k=REF_METADATA[(dataset,model)]
    with zipfile.ZipFile(REFERENCE_ZIP) as z:
        m=json.loads(z.read(k+'.json'))
        with np.load(io.BytesIO(z.read(k+'.npz')),allow_pickle=False) as npz:
            old={name:npz[name] for name in ('ya_val','yb_val','ya_test','yb_test','val_ids','test_ids')}
    X,a,b,source=q.load_pair_dataset(dataset,WORK)
    split=q.make_split(dataset,REPL_SPLIT[0],REPL_SPLIT[1],WORK,test_n=128,val_n=205)
    assert q.sha_obj({k:v for k,v in split.items() if k!='split_sha256'})==split['split_sha256']
    assert split['split_sha256']==m['split_sha256'],'Historical split differs'
    assert split['classes_a']==m['classes_a'] and split['classes_b']==m['classes_b'],'Class domains differ'
    looka={v:i for i,v in enumerate(split['classes_a'])};lookb={v:i for i,v in enumerate(split['classes_b'])}
    for role in ('val','test'):
        ids=np.asarray(split[f'{role}_ids'],dtype=int)
        assert np.array_equal(ids,old[f'{role}_ids']),f'Frozen {role} row IDs changed'
        ya=np.array([looka[str(v)] for v in a.iloc[ids]]); yb=np.array([lookb[str(v)] for v in b.iloc[ids]])
        assert np.array_equal(ya,old[f'ya_{role}']) and np.array_equal(yb,old[f'yb_{role}']),f'{dataset} {role} label mismatch'
    return {'dataset':dataset,'model':model,'status':'ALIGNED_VALIDATION_TEST_LABELS','split_sha256':m['split_sha256'],'n_rows':len(X),'n_features':X.shape[1]}

ALIGNMENT=[]
if RUN_GPU_REPLICATION:
    for dataset in REPL_DATASETS:
        # Dataset labels are shared by both models; verify each model against frozen reference anyway.
        for model in MODEL_SELECTION:
            ALIGNMENT.append(align_to_frozen(dataset,model))
else:
    print('Online alignment deferred until GPU replication is enabled; no source datasets loaded.')
(WORK/'reports'/'FROZEN_LABEL_ALIGNMENT.json').write_text(json.dumps(ALIGNMENT,indent=2))
print('Verified label alignments:',len(ALIGNMENT))

## 5. GPU replication — explicit model versions and checkpoint fingerprints

**Before enabling:** Set Colab Runtime to T4 GPU. For TabPFN-3, accept the license in your own account and add `TABPFN_TOKEN` to Colab Secrets. The notebook never exports the token. It pins model *package* versions, not an impossible reconstruction of the old runtime's entire dependency graph. New `pip freeze`, GPU data, file hashes and source/selection settings are saved.

In [ ]:
if RUN_GPU_REPLICATION:
    if not q.cuda_status().get('cuda'):raise RuntimeError('New inference requires a CUDA GPU (e.g., T4).')
    wanted={'tabicl':'2.2.0','tabpfn':'9.2.0'}
    for pkg,ver in wanted.items():
        found=q.version(pkg)
        if found!=ver:
            if not INSTALL_MODEL_PACKAGES:raise RuntimeError(f'{pkg}: installed={found}, required={ver}')
            subprocess.run([sys.executable,'-m','pip','install','-q',f'{pkg}=={ver}'],check=True)
        assert q.version(pkg)==ver,f'{pkg} package version mismatch'
    if 'tabpfn3' in MODEL_SELECTION:
        if not TABPFN_LICENSE_ACCEPTED:
            raise PermissionError('You must accept TabPFN-3 license and explicitly set TABPFN_LICENSE_ACCEPTED=True')
        if not os.environ.get('TABPFN_TOKEN') and IN_COLAB:
            from google.colab import userdata
            secret=userdata.get('TABPFN_TOKEN')
            if not secret:raise PermissionError('TABPFN_TOKEN secret not accessible')
            os.environ['TABPFN_TOKEN']=str(secret)
            del secret
    import torch
    import huggingface_hub
    ckpt_name='tabicl-classifier-v2-20260212.ckpt'
    hub=huggingface_hub.HfApi()
    pinned_commit=hub.model_info('jingang/TabICL').sha
    TABICL_PATH=Path(huggingface_hub.hf_hub_download('jingang/TabICL',filename=ckpt_name,revision=pinned_commit))
    TABICL_HASH=sha(TABICL_PATH.read_bytes())
    if TABICL_HASH!=EXPECTED_TABICL_REMOTE_SHA256:
        raise ValueError('TabICL byte SHA-256 unexpected. Do not quietly substitute a different checkpoint.')
    from tabicl import TabICLClassifier
    from tabpfn import TabPFNClassifier
    from tabpfn.constants import ModelVersion
    assert hasattr(ModelVersion,'V3')
    tb_probe=TabPFNClassifier.create_default_for_version(ModelVersion.V3)
    TABPFN_SELECTED_PATH=Path(str(tb_probe.get_params()['model_path'])).expanduser()
    print('TabICL public checkpoint SHA256:',TABICL_HASH)
    print('TabPFN V3 selected file basename:',TABPFN_SELECTED_PATH.name)

    ORIGINAL_FACTORY=q.make_factory
    def fixed_factory(model,device='cuda',seed=23):
        if model=='tabiclv2':
            return lambda: TabICLClassifier(n_estimators=2,kv_cache=False,batch_size=2,
                checkpoint_version=ckpt_name,model_path=str(TABICL_PATH),allow_auto_download=False,
                device=device,random_state=seed)
        if model=='tabpfn3':
            return lambda: TabPFNClassifier.create_default_for_version(ModelVersion.V3,
                 n_estimators=2,device=device,random_state=seed)
        return ORIGINAL_FACTORY(model,device,seed)
    q.make_factory=fixed_factory
    torch.backends.cudnn.deterministic=True
    torch.backends.cudnn.benchmark=False
    try: torch.use_deterministic_algorithms(True,warn_only=True)
    except Exception: pass
else:
    print('CPU audit mode. Model installation, authentication and inference skipped.')

## 6. Run six fresh configurations (three datasets × two models)

Each job writes **new** prediction files to the clean workspace. A comparison report calculates score and per-probability differences against the frozen reference. Floating-point GPU predictions need not be byte-for-byte identical; all disagreements are recorded. Stop if source/split/label alignment fails.

In [ ]:
import traceback
JOBS=[(d,m,REPL_SPLIT[0],REPL_SPLIT[1],REPL_CONTEXT) for d in REPL_DATASETS for m in MODEL_SELECTION]
results=[]
if RUN_GPU_REPLICATION:
    for dataset,model,seed,fold,cap in JOBS:
        path=WORK/'real_caches'/f'{dataset}__{model}__seed{seed}__fold{fold}__train{cap}.npz'
        event={'dataset':dataset,'model':model,'seed':seed,'fold':fold,'train_cap':cap}
        if path.exists():
            try:q.read_cache(path);event['status']='CACHED_REPLICATION_VERIFIED'
            except Exception as e:event.update({'status':'INVALID_REPLICATION_CACHE','error':str(e)[:160]})
        else:
            try:
                # Independent label alignment was performed before model setup.
                event.update(q.run_case(dataset,model,seed,fold,cap,root=WORK,test_n=128,val_n=205))
                if model=='tabiclv2':
                    event['configured_checkpoint_path_hash']=TABICL_HASH
                    event['configured_checkpoint_repo_revision']=pinned_commit
                    event['checkpoint_path_explicit']=True
                else:
                    event['checkpoint_path_selected_by_v3_factory']=TABPFN_SELECTED_PATH.name
                    event['checkpoint_path_explicit']=True
                    event['configured_checkpoint_path_hash']=sha(TABPFN_SELECTED_PATH.read_bytes()) if TABPFN_SELECTED_PATH.is_file() else 'FILE_NOT_FOUND_AFTER_FIT'
            except Exception as e:event.update({'status':'ERROR','exception_type':type(e).__name__,'error':str(e)[:300]})
        results.append(event)
        (WORK/'reports'/'NEW_GPU_REPLICATION_EVENTS.json').write_text(json.dumps(results,indent=2,default=str))
        print(dataset,model,event['status'])
else:
    print('GPU replication deferred: set RUN_GPU_REPLICATION=True when ready.')
print('New inference events:',len(results))

## 7. Match new results with archived probability arrays and proper scores

Numerical match is reported at **configuration** level. Old and new runs are never silently pooled. The new checkpoint file hashes are labeled as newly observed, never retroactively applied to the historical runs.

In [ ]:
comparison=[]
with zipfile.ZipFile(REFERENCE_ZIP) as z:
    for dataset,model,seed,fold,cap in JOBS:
        k=REF_METADATA[(dataset,model)]; new=WORK/'real_caches'/(Path(k).name+'.npz')
        if not new.is_file():continue
        d_new,m_new=q.read_cache(new)
        with np.load(io.BytesIO(z.read(k+'.npz')),allow_pickle=False) as f:
            old={name:f[name] for name in f.files}
        assert np.array_equal(d_new['val_ids'],old['val_ids']) and np.array_equal(d_new['test_ids'],old['test_ids'])
        assert np.array_equal(d_new['ya_test'],old['ya_test']) and np.array_equal(d_new['yb_test'],old['yb_test'])
        pnew=q.candidates(d_new,'test')['arithmetic'];pold=q.candidates(old,'test')['arithmetic']
        nnew,bnew=q.per_sample_metrics(pnew,d_new['ya_test'],d_new['yb_test'])
        nold,bold=q.per_sample_metrics(pold,old['ya_test'],old['yb_test'])
        comparison.append({'dataset':dataset,'model':model,'old_pool_nll':float(nold.mean()),
           'new_pool_nll':float(nnew.mean()),'delta_nll_new_minus_old':float(nnew.mean()-nold.mean()),
           'old_pool_brier':float(bold.mean()),'new_pool_brier':float(bnew.mean()),
           'mean_abs_probability_change':float(np.mean(np.abs(pnew-pold))),
           'max_abs_probability_change':float(np.max(np.abs(pnew-pold))),
           'historical_checkpoint_hash_status':'NOT_VERIFIED',
           'new_checkpoint_loaded_path_status':'INSPECT_NEW_EVENTS_FOR_FILE_SHA256'})
import pandas as pd
if comparison:pd.DataFrame(comparison).to_csv(WORK/'reports'/'NEW_VS_OLD_REPLICATION.csv',index=False)
print('Completed paired configuration comparisons:',len(comparison),'out of',len(JOBS))

## 8. Capture environment and securely export reports

Exclude tokens, model weights, processed pickle datasets and raw OpenML downloads from the archive. Include only the new prediction caches, split metadata, report logs, and an integrity manifest. An archived checkpoint file hash is a new-run evidence record, not the checkpoint file itself.

In [ ]:
from datetime import datetime,timezone
from zipfile import ZipFile,ZIP_DEFLATED
info={'time_utc':datetime.now(timezone.utc).isoformat(),'python':sys.version,'platform':platform.platform(),
      'packages':{},'cuda':q.cuda_status(),'source_reference_archive_sha256':sha(REFERENCE_ZIP.read_bytes()),
      'old_loaded_checkpoint_bytes_verified':False,'new_replication_executed':bool(results)}
for pkg in ('torch','tabicl','tabpfn','numpy','pandas','scipy','scikit-learn','openml','huggingface_hub'):
    info['packages'][pkg]=q.version(pkg)
(WORK/'reports'/'NEW_ENVIRONMENT_PROVENANCE.json').write_text(json.dumps(info,indent=2,default=str))
# Record all installed package versions without pip's potentially sensitive private-index URLs.
try:
    freeze=subprocess.run([sys.executable,'-m','pip','list','--format=freeze'],capture_output=True,text=True,timeout=60)
    (WORK/'reports'/'INSTALLED_PACKAGE_VERSIONS.txt').write_text(freeze.stdout if freeze.returncode==0 else 'package inventory unavailable')
except Exception:
    (WORK/'reports'/'INSTALLED_PACKAGE_VERSIONS.txt').write_text('package inventory unavailable')
try:
    p=subprocess.run(['nvidia-smi','--query-gpu=name,driver_version,memory.total','--format=csv,noheader'],capture_output=True,text=True,timeout=10)
    (WORK/'reports'/'GPU_DRIVER_INFO.txt').write_text(p.stdout if p.returncode==0 else 'not available')
except Exception:(WORK/'reports'/'GPU_DRIVER_INFO.txt').write_text('not available')
# The source fingerprint is a SHA-256 of the exact embedded frozen inference engine.
(WORK/'reports'/'EMBEDDED_ENGINE_SHA256.txt').write_text(sha(ENGINE_SOURCE.encode()))
from pathlib import Path
output=WORK/'CoReTFM_NEW_CLEAN_REPLICATION_REPORTS.zip'
allowed=[]
for folder in ('reports','splits','real_caches'):
    for f in sorted((WORK/folder).rglob('*')):
        if f.is_file() and f.suffix.lower() not in ('.zip','.pkl','.ckpt','.pt','.pth','.safetensors','.token'):
            allowed.append(f)
manifest_out=[]
with ZipFile(output,'w',ZIP_DEFLATED,compresslevel=5) as z:
    for f in allowed:
        rel=f.relative_to(WORK).as_posix()
        z.write(f,rel)
        manifest_out.append({'path':rel,'sha256':sha(f.read_bytes()),'bytes':f.stat().st_size})
    z.writestr('NEW_REPLICATION_SHA256_MANIFEST.json',json.dumps(manifest_out,indent=2))
with ZipFile(output) as z:
    assert z.testzip() is None
    for rec in manifest_out:assert sha(z.read(rec['path']))==rec['sha256']
print('Verified report ZIP:',output,'members:',len(manifest_out),'size MB:',round(output.stat().st_size/2**20,2))
if IN_COLAB and DOWNLOAD_RESULT_ZIP:
    from google.colab import files
    files.download(str(output))

### What counts as a successful replication?

- 160/160 historical caches remain immutable, and all 40 frozen split specifications are unchanged.
- A source dataset can be independently retrieved with the same **ordered** validation/test labels (mismatch is a hard failure).
- The exact **newly configured** TabICLv2 checkpoint is identified by path, HF revision and byte SHA-256; TabPFN-3 uses an explicit V3 factory and records its selected local checkpoint file hash after successful fitting.
- Six new `.npz` caches and six paired old-vs-new results exist. Any numerical divergence is reported, **not concealed**.
- This does not retrospectively authenticate checkpoints used by the original experiments or guarantee exact GPU bitwise reproducibility.